
## Folder 04 — File 05: Silver Payments:


**Transformation rules:**

- Trim payment IDs, order IDs, transaction IDs, and payment methods.
- Standardize payment_status and payment_method to uppercase.
- Convert amount to DECIMAL(12,2).
- Convert payment_timestamp to TIMESTAMP.
- Validate payment status against SUCCESS, PENDING, FAILED, and REFUNDED.
- Quarantine missing identifiers, invalid amounts, timestamps, and statuses.
- Deduplicate using payment_id.

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"

BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_payments"
)

SILVER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_payments"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_payments"
)

ALLOWED_PAYMENT_STATUSES = [
    "SUCCESS",
    "PENDING",
    "FAILED",
    "REFUNDED"
]


## 3. Create Silver schema

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{SILVER_SCHEMA}`"
)

print("Silver schema is ready.")


## 4. Read Bronze payments

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise Exception(
        f"Bronze table not found: {BRONZE_TABLE}. "
        "Run Folder 03 first."
    )

bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze payment records: {bronze_df.count()}")

display(bronze_df.limit(10))


## 5. Clean and standardize payment columns

cleaned_df = (
    bronze_df

    .withColumn(
        "payment_id",
        F.trim(F.col("payment_id").cast("string"))
    )
    .withColumn(
        "order_id",
        F.trim(F.col("order_id").cast("string"))
    )
    .withColumn(
        "payment_method",
        F.upper(F.trim(F.col("payment_method").cast("string")))
    )
    .withColumn(
        "payment_status",
        F.upper(F.trim(F.col("payment_status").cast("string")))
    )
    .withColumn(
        "transaction_id",
        F.trim(F.col("transaction_id").cast("string"))
    )

    # Convert amount into DECIMAL(12,2)

    .withColumn(
        "amount",
        F.expr("try_cast(amount AS DECIMAL(12,2))")
    )

    # Convert payment timestamp

    .withColumn(
        "payment_timestamp",
        F.expr("try_cast(payment_timestamp AS TIMESTAMP)")
    )
)

print("Payment standardization completed.")

display(cleaned_df.limit(10))


## 6. Identify invalid payment records

invalid_condition = (
    F.col("payment_id").isNull()
    | (F.col("payment_id") == "")

    | F.col("order_id").isNull()
    | (F.col("order_id") == "")

    | F.col("payment_method").isNull()
    | (F.col("payment_method") == "")

    | F.col("payment_status").isNull()
    | (~F.col("payment_status").isin(ALLOWED_PAYMENT_STATUSES))

    | F.col("amount").isNull()
    | (F.col("amount") < 0)

    | F.col("payment_timestamp").isNull()
)

invalid_df = (
    cleaned_df
    .filter(invalid_condition)
    .withColumn(
        "_quarantine_reason",
        F.concat_ws(
            "; ",

            F.when(
                F.col("payment_id").isNull()
                | (F.col("payment_id") == ""),
                F.lit("Missing payment_id")
            ),

            F.when(
                F.col("order_id").isNull()
                | (F.col("order_id") == ""),
                F.lit("Missing order_id")
            ),

            F.when(
                F.col("payment_method").isNull()
                | (F.col("payment_method") == ""),
                F.lit("Missing payment_method")
            ),

            F.when(
                F.col("payment_status").isNull()
                | (~F.col("payment_status").isin(ALLOWED_PAYMENT_STATUSES)),
                F.lit("Invalid payment_status")
            ),

            F.when(
                F.col("amount").isNull(),
                F.lit("Missing or invalid amount")
            ),

            F.when(
                F.col("amount") < 0,
                F.lit("Negative payment amount")
            ),

            F.when(
                F.col("payment_timestamp").isNull(),
                F.lit("Missing or invalid payment_timestamp")
            )
        )
    )
)

valid_df = cleaned_df.filter(~invalid_condition)

print(f"Valid payment records: {valid_df.count()}")
print(f"Invalid payment records: {invalid_df.count()}")


## 7. Remove duplicate payment records

deduplication_window = (
    Window
    .partitionBy("payment_id")
    .orderBy(
        F.col("_ingestion_timestamp").desc_nulls_last(),
        F.col("_source_file").desc_nulls_last()
    )
)

deduplicated_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(deduplication_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

print(
    f"Records after deduplication: "
    f"{deduplicated_df.count()}"
)


## 8. Save invalid records to quarantine

(
    invalid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Quarantine table created: {QUARANTINE_TABLE}")


## 9. Save clean records to Silver Delta table

(
    deduplicated_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table created: {SILVER_TABLE}")


## 10. Validate Silver payment records

silver_df = spark.table(SILVER_TABLE)

total_silver_records = silver_df.count()

duplicate_payment_groups = (
    silver_df
    .groupBy("payment_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_silver_records = (
    silver_df
    .filter(
        F.col("payment_id").isNull()
        | F.col("order_id").isNull()
        | F.col("payment_method").isNull()
        | F.col("payment_status").isNull()
        | (~F.col("payment_status").isin(ALLOWED_PAYMENT_STATUSES))
        | F.col("amount").isNull()
        | (F.col("amount") < 0)
        | F.col("payment_timestamp").isNull()
    )
    .count()
)

print("=" * 50)
print("SILVER PAYMENT VALIDATION")
print("=" * 50)

print(f"Silver records: {total_silver_records}")
print(f"Duplicate payment groups: {duplicate_payment_groups}")
print(f"Invalid Silver records: {invalid_silver_records}")

if duplicate_payment_groups > 0 or invalid_silver_records > 0:
    raise Exception("Silver payment validation failed.")

print("Silver payment validation passed.")


## 11. Payment status analysis

display(
    silver_df
    .groupBy("payment_status")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount")
    )
    .orderBy("payment_status")
)


## 12. Payment method analysis

display(
    silver_df
    .groupBy("payment_method")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount")
    )
    .orderBy(F.desc("total_amount"))
)


## 13. Display final Silver payment data

display(
    silver_df.orderBy("payment_timestamp").limit(20)
)


## 14. Final summary

print("=" * 50)
print("PAYMENT SILVER TRANSFORMATION COMPLETED")
print("=" * 50)

print(f"Bronze table     : {BRONZE_TABLE}")
print(f"Silver table     : {SILVER_TABLE}")
print(f"Quarantine table : {QUARANTINE_TABLE}")

print(f"Bronze records   : {bronze_df.count()}")
print(f"Silver records   : {total_silver_records}")
print(f"Invalid records  : {invalid_df.count()}")

print("=" * 50)
